# Практика: два запуска K-means

Работайте сверху вниз из папки `05-clustering-kmeans` в копии репозитория. Нужны `pandas`, `matplotlib`, `scikit-learn`. В Colab достаточно открыть этот ноутбук: данные Iris входят в `scikit-learn`.

Сначала предположите, будут ли два разных начала приводить к одинаковым группам. Затем проверьте предположение на восьми точках и на ирисах. Номера кластеров условны.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris

points = pd.DataFrame([(0, 0), (0, 2), (1, 0), (1, 2),
                       (4, 0), (4, 2), (5, 0), (5, 2)], columns=['x', 'y'])
print(points)


## 1. Два разных начала на восьми точках

Сравните вертикальное и горизонтальное разбиения при одинаковом числе групп. `n_init=1` означает один запуск с указанными центрами. Запишите, какие точки попали в каждую группу и какое значение `inertia_` получила модель.


In [ ]:
starts = {'слева и справа': [[0, 0], [4, 0]],
          'снизу и сверху': [[0, 0], [0, 2]]}
for label, centers in starts.items():
    model = KMeans(n_clusters=2, init=pd.DataFrame(centers).to_numpy(),
                   n_init=1, random_state=0).fit(points)
    print(label, 'центры:', model.cluster_centers_.round(2).tolist(),
          'группы:', model.labels_.tolist(), 'J:', round(model.inertia_, 2))


**Самостоятельно.** Замените вторую пару центров на `(1, 0)` и `(5, 0)`. Сначала предположите, изменится ли конечное разбиение. Проверьте по `labels_` и `inertia_`. Объясните, почему сравнивать суммы допустимо здесь: точки, число групп и масштаб одинаковы.

**Самопроверка.** В исходных запусках суммы равны 10 и 34. Номера групп можно переставить; сравнивать нужно состав групп и центры.


## 2. Ирисы: таблица и диаграмма

Модель увидит только длину и ширину лепестка. Известный вид нужен для интерпретации уже полученных групп, но не для обучения. На диаграмме окраска показывает найденный кластер.


In [ ]:
iris = load_iris()
flowers = pd.DataFrame(iris.data[:, 2:4], columns=['length_cm', 'width_cm'])
initial = flowers.iloc[[0, 50, 100]].to_numpy()
model = KMeans(n_clusters=3, init=initial, n_init=1, random_state=0)
flowers['cluster'] = model.fit_predict(flowers[['length_cm', 'width_cm']])
print(flowers.groupby('cluster')[['length_cm', 'width_cm']].mean().round(2))
fig, ax = plt.subplots(figsize=(7, 4))
for group, part in flowers.groupby('cluster'):
    ax.scatter(part['length_cm'], part['width_cm'], s=18, alpha=.7, label=f'Группа {group}')
ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
           marker='X', s=130, c='black', label='Центры')
ax.set(xlabel='Длина лепестка, см', ylabel='Ширина лепестка, см', title='Ирисы после K-means')
ax.legend()
plt.show()


**Самостоятельно.** Возьмите другие три строки для начальных центров, например `flowers.iloc[[10, 60, 110]][['length_cm', 'width_cm']].to_numpy()`. Постройте модель заново. Запишите сумму квадратов расстояний, размеры групп и положение центров. Что поменялось? Если численные номера групп поменялись, сравните также их состав, а не только номера.

**Дополнительно.** Попробуйте `KMeans(n_clusters=3, n_init=10, random_state=0)` и сравните `inertia_`. Почему меньшая сумма не доказывает совпадения групп с видами?

**Самопроверка.** Проверяйте, что `len(flowers) == 150`, суммарный размер групп равен 150 и виды не входят в признаки. `inertia_` оценивает компактность групп по двум числовым признакам; биологический смысл она не измеряет.
